# Baseline Forecasting Models

This notebook establishes baseline performance for one-month-ahead bilateral trade-flow forecasting across South Korea, Slovakia, and the USA. Two methods are evaluated: the naive model, which uses the most recent observed value, and the seasonal-naive model, which uses the value from the same month in the previous year.

Both models use training data available through February 2026 and are evaluated against the held-out March 2026 targets. Their results provide reference points for assessing whether more complex forecasting models improve predictive accuracy.

## 1. Load Forecasting Data

Load the training datasets and held-out target values prepared in `08_forecasting_preparation.ipynb`. These files provide a consistent training cutoff and evaluation month for all baseline models.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_loading import (
    load_forecasting_data,
    SERIES_KEYS,
    TARGET,
)
from src.evaluation import evaluate_forecasts, save_model_results

import numpy as np
import pandas as pd

training_data, test_data = load_forecasting_data()

for country in training_data:
    print(
        f"{country}: {len(training_data[country]):,} training rows, "
        f"{len(test_data[country]):,} test series"
    )

South Korea: 2,374,832 training rows, 27,823 test series
Slovakia: 2,826,210 training rows, 21,973 test series
USA: 3,880,587 training rows, 35,843 test series


### 2. Generate baseline forecasts
This cell creates naive forecasts using February 2026 values and seasonal-naive forecasts using March 2025 values. Both are compared against the held-out March 2026 targets.

In [ ]:

baseline_predictions = {}
SEASONAL_LAG = 12
TEST_MONTH = pd.Timestamp("2026-03-01")

for country, train_df in training_data.items():
    test_df = test_data[country]

    # Naive forecast: most recent observed value
    last_values = (
        train_df.sort_values("Month")
        .groupby(SERIES_KEYS, as_index=False, sort=False)
        .tail(1)[SERIES_KEYS + [TARGET]]
        .rename(columns={TARGET: "naive_prediction"})
    )

    # Seasonal-naive forecast: same month one year earlier
    seasonal_month = TEST_MONTH - pd.DateOffset(months=SEASONAL_LAG)
    seasonal_values = (
        train_df.loc[
            train_df["Month"].eq(seasonal_month),
            SERIES_KEYS + [TARGET],
        ]
        .rename(columns={TARGET: "seasonal_naive_prediction"})
    )

    predictions = (
        test_df.merge(
            last_values,
            on=SERIES_KEYS,
            how="left",
            validate="one_to_one",
        )
        .merge(
            seasonal_values,
            on=SERIES_KEYS,
            how="left",
            validate="one_to_one",
        )
    )

    assert predictions["naive_prediction"].notna().all()
    assert predictions["seasonal_naive_prediction"].notna().all()

    baseline_predictions[country] = predictions

    print(
        f"{country}: {len(predictions):,} series forecasted; "
        "naive and seasonal-naive predictions available."
    )

South Korea: 27,823 series forecasted; naive and seasonal-naive predictions available.
Slovakia: 21,973 series forecasted; naive and seasonal-naive predictions available.
USA: 35,843 series forecasted; naive and seasonal-naive predictions available.


## 3.Evaluation and Results

Evaluate the naive and seasonal-naive forecasts using MAE, RMSE, and sMAPE. Save the results in the standardized format defined in `src/evaluation.py` for comparison with other forecasting models.


In [5]:
baseline_results = []

for country, predictions in baseline_predictions.items():
    for model_name, prediction_column in [
        ("Naive", "naive_prediction"),
        ("Seasonal Naive", "seasonal_naive_prediction"),
    ]:
        baseline_results.append(
            evaluate_forecasts(
                country=country,
                model_name=model_name,
                actual=predictions["Value"],
                predicted=predictions[prediction_column],
                forecast_month="2026-03",
            )
        )

baseline_results_df = save_model_results(
    baseline_results,
    model_name="Baseline",
)

baseline_results_df

Saved model results to: C:\Transfer\InterdisciplinaryProject\results\baseline_results.csv


,Country,Model,Forecast Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,Naive,2026-03,21973,2.244378e+05,1.829875e+06,79.873091
1,Slovakia,Seasonal Naive,2026-03,21973,2.869592e+05,2.275405e+06,85.384997
2,South Korea,Naive,2026-03,27823,1.393658e+06,2.299927e+07,74.623863
3,South Korea,Seasonal Naive,2026-03,27823,1.984834e+06,4.983757e+07,78.922648
4,USA,Naive,2026-03,35843,3.091994e+06,3.113052e+07,75.879247
5,USA,Seasonal Naive,2026-03,35843,6.066488e+06,1.352464e+08,77.520092



## Conclusion

The naive and seasonal-naive models establish the baseline performance for one-month-ahead trade-flow forecasting for March 2026. Across all three countries, the naive model outperforms the seasonal-naive model on all three evaluation metrics: MAE, RMSE, and sMAPE.


